In [1]:
import pandas as pd
import pickle
import pandas as pd
import numpy as np
import os


from cobrame.solve.algorithms import binary_search
from cobrame.solve.symbolic import compile_expressions
from cobra import Reaction, Metabolite

import cobra 
import cobrame 
import Bio 
import pickle 
import ecolime
from os.path import dirname, abspath, join


/home/jkrishnan/miniconda/envs/cobrame_env/lib/python3.6/site-packages/cobra/io/sbml3.py:24: UserWarning: Install lxml for faster SBML I/O
  warn("Install lxml for faster SBML I/O")
/home/jkrishnan/miniconda/envs/cobrame_env/lib/python3.6/site-packages/cobra/io/__init__.py:12: UserWarning: cobra.io.sbml requires libsbml
  warn("cobra.io.sbml requires libsbml")


In [2]:
ecoli_dir = dirname(abspath(ecolime.__file__)) 
model_file = join(ecoli_dir, "me_models", "iJL1678b.pickle") 


with open(model_file, "rb") as f:
    base_me = pickle.load(f)

In [3]:
ntr1 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_NtrC-1_genes.csv')
ntr2 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_NtrC-2_genes.csv')
gadx = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_GadX_genes.csv')
arca1 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_ArcA-1_genes.csv')
arca2 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_ArcA-2_genes.csv')
fnr1 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_Fnr-1_genes.csv')
fnr2 = pd.read_csv('imodulon_genes/e_coli_precise_mg1655_Fnr-2_genes.csv')
ribosomal_genes = pd.read_csv('imodulon_genes/Ribosome_Genes.csv',header=None)

ntr1_genes = ntr1['gene_locus'].unique().tolist()
ntr2_genes = ntr2['gene_locus'].unique().tolist()
gadx_genes = gadx['gene_locus'].unique().tolist()
arca1_genes = arca1['gene_locus'].unique().tolist()
arca2_genes = arca2['gene_locus'].unique().tolist()
fnr1_genes = fnr1['gene_locus'].unique().tolist()
fnr2_genes = fnr2['gene_locus'].unique().tolist()
ribosomal_genes = ribosomal_genes[0].tolist()

In [4]:
tpm_metadata = pd.read_excel(
    "sample_table_TPMcolumn_to_nitrogen_source.xlsx",
    engine="openpyxl"
)
tpm = pd.read_csv('tpm_qc_nitrogen42_labeled.csv')
# ----------------------------------------------------------
# Average replicate TPMs, one column per nitrogen source
# ----------------------------------------------------------

# tpm columns are named '<source>__<tpm_column>', so map each
# sample back to its nitrogen source through the metadata table
col_to_source = dict(
    zip(
        tpm_metadata["tpm_column"],
        tpm_metadata["nitrogen_source"]
    )
)

tpm_indexed = tpm.set_index("gene")

sample_to_source = {
    col: col_to_source[col.split("__", 1)[-1]]
    for col in tpm_indexed.columns
}

# mean across all replicates (and conditions) of a given source
tpm_by_source = (
    tpm_indexed
    .rename(columns=sample_to_source)
    .T
    .groupby(level=0)
    .mean()
    .T
)

# keep the metadata ordering of the sources
source_order = (
    tpm_metadata["nitrogen_source"]
    .drop_duplicates()
    .tolist()
)

tpm_by_source = tpm_by_source[source_order]

print(
    "Samples:", tpm_indexed.shape[1],
    "-> nitrogen sources:", tpm_by_source.shape[1]
)

tpm_by_source

Samples: 42 -> nitrogen sources: 20


,NH4Cl,Adenine,L-Cys,Cytosine,Cytidine,Putrescine,L-Pro,Adenosine,L-Asn,GlcNAc,Neu5Ac,Glycine,L-Orn,L-Asp,D-Ala,L-Ala,L-Arg,GABA,L-Ser,Guanosine
gene,,,,,,,,,,,,,,,,,,,,
b0002,1452.739949,807.712691,3668.477429,1189.297607,1416.821032,726.247342,1571.540602,1210.587389,1145.105477,1812.716410,2838.010797,1992.237003,423.789321,694.788479,991.174448,1868.830721,841.130657,315.286543,1034.438597,738.610713
b0003,979.674494,775.305099,2201.027297,991.076416,1030.883920,618.924735,1610.111252,992.712237,964.269949,1389.700289,2231.697049,1785.294203,383.977124,629.376057,935.600696,1632.877413,1090.957935,340.283598,1236.119614,580.452964
b0004,880.886520,670.279724,1699.636408,800.664448,902.391655,526.487113,1205.251439,820.765173,841.902267,1169.952998,1875.229575,1309.006182,292.153874,443.250831,804.668650,1249.369512,878.416367,282.209421,1432.264653,407.326130
b0005,79.608723,47.842034,125.571824,47.670575,48.909666,40.525193,85.350646,57.546487,57.085822,72.432807,121.065434,57.064159,21.801175,26.471742,61.498879,122.130734,98.409443,30.229615,101.661574,20.819400
b0006,60.998350,59.135531,61.922406,57.876241,59.964104,53.030248,45.956731,62.201281,56.490255,67.933778,52.642066,54.723359,41.312481,50.651329,58.747963,47.816220,23.840951,60.393801,48.908585,19.844364
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
b4399,17.034449,22.362379,44.781272,27.382784,23.508501,26.186170,18.691615,23.477313,15.504157,18.739157,18.419038,19.454019,18.911224,17.176853,19.131592,24.020171,34.352227,26.404199,20.252759,12.559343
b4400,15.248015,9.300182,23.002389,12.118988,7.023145,10.395325,10.421499,7.438150,9.854366,7.455269,10.175140,14.491690,14.358067,11.926679,12.279938,14.075453,17.518502,14.964347,10.081995,18.513344
b4401,377.959326,469.681872,875.098110,590.704228,579.929756,714.752252,889.754727,1232.686226,1822.371074,1419.331009,1152.083984,1508.244254,2328.505163,1540.263584,656.495160,786.460644,1047.098080,764.483374,669.108789,698.856090


In [5]:
imodulon_genes = {}
imod_dir = "/home/jkrishnan/m_models/ecoli/jongoh_nitrogen/imod_genes"
imods_to_consider = ['NtrC-1','NtrC-2','NtrC-3','GadX-1','GadX-2','ArcA-1','ArcA-2','Fnr-1','Fnr-2','Fnr-3']

for fname in os.listdir(imod_dir):

    if fname.endswith(".pkl"):

        sector = os.path.splitext(fname)[0]

        with open(
            os.path.join(imod_dir, fname),
            "rb"
        ) as f:
            if(sector in imods_to_consider):
                imodulon_genes[sector] = pickle.load(f)


# -----------------------------
# Add ribosome sector explicitly
# -----------------------------

imodulon_genes["Ribosome"] = ribosomal_genes


# -----------------------------
# Transcript fractions
# -----------------------------

sector_fraction = {}

for sector, genes in imodulon_genes.items():

    genes_in_tpm = [
        g for g in genes
        if g in tpm_by_source.index
    ]

    sector_fraction[sector] = (
        tpm_by_source.loc[genes_in_tpm].sum(axis=0)
        / tpm_by_source.sum(axis=0)
    )

sector_fraction = pd.DataFrame(
    sector_fraction
).T


# -----------------------------
# Translation reactions in ME model
# -----------------------------

sector_prots = {}

for sector, genes in imodulon_genes.items():

    sector_prots[sector] = [
        "translation_" + gene
        for gene in genes
        if "translation_" + gene in base_me.reactions
    ]

    print(
        sector,
        "Genes:", len(genes),
        "Proteins represented:",
        len(sector_prots[sector])
    )

NtrC-3 Genes: 62 Proteins represented: 41
GadX-1 Genes: 21 Proteins represented: 5
ArcA-1 Genes: 81 Proteins represented: 44
NtrC-2 Genes: 9 Proteins represented: 5
Fnr-1 Genes: 56 Proteins represented: 33
Fnr-3 Genes: 43 Proteins represented: 18
GadX-2 Genes: 11 Proteins represented: 2
ArcA-2 Genes: 28 Proteins represented: 26
NtrC-1 Genes: 13 Proteins represented: 10
Fnr-2 Genes: 52 Proteins represented: 16
Ribosome Genes: 58 Proteins represented: 55


In [6]:
sector_fraction

,NH4Cl,Adenine,L-Cys,Cytosine,Cytidine,Putrescine,L-Pro,Adenosine,L-Asn,GlcNAc,Neu5Ac,Glycine,L-Orn,L-Asp,D-Ala,L-Ala,L-Arg,GABA,L-Ser,Guanosine
NtrC-3,0.021064,0.015404,0.007517,0.058553,0.075173,0.048904,0.048006,0.053155,0.038262,0.045664,0.047216,0.041560,0.041506,0.030810,0.032245,0.033846,0.040442,0.034363,0.017127,0.073060
GadX-1,0.002086,0.002780,0.018637,0.003482,0.004923,0.006753,0.020290,0.049754,0.103851,0.069640,0.058825,0.088163,0.090536,0.097944,0.023404,0.078508,0.064121,0.041672,0.049764,0.030942
ArcA-1,0.014362,0.013904,0.018949,0.017037,0.019750,0.022453,0.017657,0.014880,0.010361,0.011821,0.010527,0.012182,0.017571,0.011580,0.015714,0.012964,0.014071,0.030890,0.008900,0.049108
NtrC-2,0.001245,0.001080,0.000502,0.015577,0.025965,0.017569,0.020185,0.025457,0.018014,0.022577,0.013266,0.021008,0.019077,0.016625,0.010991,0.018084,0.018020,0.017923,0.008004,0.031271
Fnr-1,0.005208,0.012624,0.009200,0.009655,0.007290,0.008490,0.005045,0.009154,0.007135,0.010113,0.009617,0.006824,0.004932,0.006250,0.013812,0.006041,0.006241,0.029732,0.021619,0.017884
Fnr-3,0.004144,0.004332,0.008955,0.003503,0.004062,0.009384,0.005495,0.002705,0.002926,0.002772,0.002843,0.004525,0.006137,0.006488,0.011924,0.002532,0.005430,0.052832,0.021879,0.030858
GadX-2,0.001067,0.001328,0.006575,0.001738,0.002174,0.003302,0.007320,0.016517,0.037150,0.023403,0.020614,0.024353,0.032103,0.034139,0.007861,0.023592,0.019077,0.012846,0.015698,0.008293
ArcA-2,0.018931,0.024682,0.018880,0.025679,0.027076,0.019991,0.015531,0.021768,0.014867,0.015831,0.022042,0.016596,0.011790,0.010547,0.019057,0.014129,0.011342,0.006985,0.010564,0.006967
NtrC-1,0.000136,0.000114,0.000510,0.001369,0.002060,0.002166,0.003925,0.001870,0.001799,0.001372,0.002093,0.002051,0.006231,0.004120,0.000542,0.001482,0.006652,0.006046,0.000281,0.026393
Fnr-2,0.003581,0.004690,0.004306,0.002943,0.002791,0.009064,0.003606,0.002835,0.002890,0.002518,0.002163,0.003079,0.004298,0.003236,0.010284,0.003491,0.005105,0.033097,0.018654,0.009769


In [7]:
def create_protein_sum(me, group_name, translation_list):

    rxn_sum = Reaction(group_name)
    me.add_reaction(rxn_sum)

    cons = Metabolite("sum_" + group_name)
    cons._constraint_sense = "E"
    cons._bound = 0

    me.add_metabolites([cons])

    for rid in translation_list:

        if rid in me.reactions:

            gene = rid.replace(
                "translation_",
                ""
            )

            protein = me.metabolites.get_by_id(
                "protein_" + gene
            )

            mw = protein.formula_weight

            me.reactions.get_by_id(
                rid
            ).add_metabolites(
                {cons: mw},
                combine=True
            )

    rxn_sum.add_metabolites({
        cons: -1.0
    })
    
    
def set_protein_ratio(
    me,
    sector_name,
    ribosome_name,
    phi_sector,
    phi_ribosome
):

    cons = Metabolite(
        sector_name + "_ratio"
    )

    cons._constraint_sense = "G"
    cons._bound = 0

    me.add_metabolites([cons])

    # phi_r * V_sector
    me.reactions.get_by_id(
        sector_name
    ).add_metabolites({
        cons: phi_ribosome
    })

    # - phi_sector * V_r
    me.reactions.get_by_id(
        ribosome_name
    ).add_metabolites({
        cons: -phi_sector
    })

In [8]:
# ----------------------------------------------------------
# Nitrogen sources
# ----------------------------------------------------------

nitrogen_exchange_map = {
    'NH4Cl': 'EX_nh4_e',
    'Adenine': 'EX_ade_e',
    'L-Cys': 'EX_cys__L_e',
    'Cytosine': 'EX_csn_e',
    'Cytidine': 'EX_cytd_e',
    'Putrescine': 'EX_ptrc_e',
    'L-Pro': 'EX_pro__L_e',
    'Adenosine': 'EX_adn_e',
    'L-Asn': 'EX_asn__L_e',
    'GlcNAc': 'EX_acgam_e',
    'Glycine': 'EX_gly_e',
    'L-Orn': 'EX_orn_e',
    'L-Asp': 'EX_asp__L_e',
    'D-Ala': 'EX_ala__D_e',
    'L-Ala': 'EX_ala__L_e',
    'L-Arg': 'EX_arg__L_e',
    'GABA': 'EX_4abut_e',
    'L-Ser': 'EX_ser__L_e',
    'Guanosine': 'EX_gsn_e'
}


# ----------------------------------------------------------
# Number of N atoms in each source
# ----------------------------------------------------------

nitrogen_atoms = {
    'NH4Cl': 1,
    'Adenine': 5,
    'L-Cys': 1,
    'Cytosine': 3,
    'Cytidine': 3,
    'Putrescine': 2,
    'L-Pro': 1,
    'Adenosine': 5,
    'L-Asn': 2,
    'GlcNAc': 1,
    'Glycine': 1,
    'L-Orn': 2,
    'L-Asp': 1,
    'D-Ala': 1,
    'L-Ala': 1,
    'L-Arg': 4,
    'GABA': 1,
    'L-Ser': 1,
    'Guanosine': 5
}


# ----------------------------------------------------------
# Nitrogen-source groups
# ----------------------------------------------------------

group_A_subs = [
    'NH4Cl',
    'Adenine',
    'L-Cys'
]

group_B_subs = [
    'Cytosine',
    'Cytidine'
]

group_C_subs = [
    'Putrescine',
    'L-Pro',
    'Adenosine',
    'L-Asn',
    'GlcNAc',
    'Glycine',
    'L-Orn',
    'L-Asp',
    'D-Ala',
    'L-Ala',
    'L-Arg'
]

group_D_subs = [
    'GABA',
    'L-Ser',
    'Guanosine'
]


In [9]:
observed_growth_rates = pd.read_excel(
    'sample_table_TPMcolumn_to_nitrogen_source.xlsx',
    engine='openpyxl'
)

observed_growth_rates
observed_growth_rates_avg = (
    observed_growth_rates
    .groupby("nitrogen_source")["Growth Rate (1/hr)"]
    .mean()
)

observed_growth_rates_avg

nitrogen_source
Adenine       0.5530
Adenosine     0.6260
Cytidine      0.4870
Cytosine      0.2485
D-Ala         0.4095
GABA          0.1775
GlcNAc        0.4610
Glycine       0.2380
Guanosine     0.2055
L-Ala         0.4035
L-Arg         0.2445
L-Asn         0.4185
L-Asp         0.2790
L-Cys         0.1240
L-Orn         0.1590
L-Pro         0.1350
L-Ser         0.3745
NH4Cl         0.7415
Neu5Ac        0.3620
Putrescine    0.4135
Name: Growth Rate (1/hr), dtype: float64

In [10]:
base_me.reactions.get_by_id('ATPM').reaction

'atp_c + h2o_c --> adp_c + h_c + pi_c'

In [11]:
base_me.reactions.get_by_id('GAM').reaction

'34.98 atp_c + 34.98 h2o_c --> 34.98 adp_c + 34.98 h_c + 34.98 pi_c'

In [ ]:
from cobrame.solve.algorithms import solve_at_growth_rate

growth_rates = {}
flux_dict = {}
translation_flux_dict = {}
carbon_uptake = {}
uptake_dict = {}

for nitrogen_source, exchange_id in nitrogen_exchange_map.items():

    print("\n", nitrogen_source)

    # fresh model
    me = pickle.loads(
        pickle.dumps(base_me)
    )

    # -------------------------------
    # Medium
    # -------------------------------

    for ex in nitrogen_exchange_map.values():
        if ex in me.reactions:
            me.reactions.get_by_id(ex).lower_bound = 0

    me.reactions.get_by_id(
        "EX_glc__D_e"
    ).lower_bound = -10

    me.reactions.get_by_id(
        exchange_id
    ).lower_bound = (
        -8 / nitrogen_atoms[nitrogen_source]
    )

    # -------------------------------
    # Ribosomal protein sector
    # -------------------------------

    create_protein_sum(
        me,
        "protein_Ribosome",
        sector_prots["Ribosome"]
    )

    phi_ribosome = sector_fraction.loc[
        "Ribosome",
        nitrogen_source
    ]

    # -------------------------------
    # Other protein sectors
    # -------------------------------

    for sector, translation_rxns in sector_prots.items():

        if sector == "Ribosome":
            continue


        # skip sectors with no proteins represented in ME model
        if len(translation_rxns) == 0:
            print("Skipping", sector, "- no proteins represented")
            continue

        sector_name = "protein_" + sector

        create_protein_sum(
            me,
            sector_name,
            translation_rxns
        )

        phi_sector = sector_fraction.loc[
            sector,
            nitrogen_source
        ]

        set_protein_ratio(
            me,
            sector_name,
            "protein_Ribosome",
            phi_sector,
            phi_ribosome
        )

    # -------------------------------
    # Objective: minimum total carbon uptake at the observed mu
    # -------------------------------

    # drop the default biomass_dilution objective; mu is pinned by
    # solve_at_growth_rate, so that objective does nothing anyway
    for rxn in me.reactions:
        if rxn.objective_coefficient != 0:
            rxn.objective_coefficient = 0.

    # weight every open carbon-containing exchange by its carbon
    # atoms. uptake is a negative flux and cobrame always maximizes,
    # so maximizing sum(C_i * v_EX_i) minimizes the carbon drawn
    # from the medium
    carbon_exchanges = {}

    for rxn in me.reactions:

        if not rxn.id.startswith("EX_"):
            continue

        # only sources the medium actually makes available
        if rxn.lower_bound >= 0:
            continue

        try:
            n_c = sum(
                met.elements.get("C", 0)
                for met in rxn.metabolites
            )
        except Exception:
            continue

        if n_c == 0:
            continue

        carbon_exchanges[rxn.id] = n_c
        rxn.objective_coefficient = float(n_c)

    # -------------------------------
    # Solve
    # -------------------------------
    mu_obs = float(
        observed_growth_rates_avg[nitrogen_source]
    )

    solve_at_growth_rate(
        me,
        mu_obs,
        solver="soplex"
    )

    if me.solution.status != "optimal":
        raise ValueError(
            "Model infeasible at observed growth rate"
        )

    growth_rates[nitrogen_source] = mu_obs

    flux_dict[nitrogen_source] = (
        me.solution.x_dict.copy()
    )

    translation_flux_dict[nitrogen_source] = (
        me.get_translation_flux().copy()
    )

    # mmol carbon / gDW / h consumed from the medium
    carbon_uptake[nitrogen_source] = -sum(
        n_c * me.solution.x_dict[rid]
        for rid, n_c in carbon_exchanges.items()
    )

    # -------------------------------
    # Store the substrate uptakes
    # -------------------------------

    # exchange fluxes are negative for uptake; flip the sign so the
    # stored numbers are consumption rates (mmol / gDW / h)
    glc_uptake = -me.solution.x_dict["EX_glc__D_e"]
    n_src_uptake = -me.solution.x_dict[exchange_id]

    uptake_dict[nitrogen_source] = {
        "growth_rate": mu_obs,
        "glucose_exchange": "EX_glc__D_e",
        "glucose_uptake": glc_uptake,
        "n_source_exchange": exchange_id,
        "n_source_uptake": n_src_uptake,
        "n_source_uptake_lb": (
            8. / nitrogen_atoms[nitrogen_source]
        ),
        "N_uptake": (
            n_src_uptake
            * nitrogen_atoms[nitrogen_source]
        ),
        "C_uptake_total": carbon_uptake[nitrogen_source],
    }

    print(
        "Growth =", mu_obs,
        "| glc =", glc_uptake,
        "|", nitrogen_source, "=", n_src_uptake,
        "| C uptake =", carbon_uptake[nitrogen_source]
    )



In [ ]:
growth_df = pd.DataFrame.from_dict(
    growth_rates,
    orient="index",
    columns=["max_growth_rate"]
)

fluxes = pd.DataFrame(
    flux_dict
)

translation_fluxes = pd.DataFrame(
    translation_flux_dict
)

# one row per nitrogen source, columns as stored above
uptake_df = pd.DataFrame.from_dict(
    uptake_dict,
    orient="index"
)

uptake_df.index.name = "nitrogen_source"

display(growth_df)
display(uptake_df)

In [ ]:
growth_df.to_csv('fluxes/imod_constrained_growth.csv')
fluxes.to_csv('fluxes/imod_constrained_fluxes.csv')
translation_fluxes.to_csv('fluxes/imod_constrained_translation_fluxes.csv')
uptake_df.to_csv('fluxes/imod_constrained_uptakes.csv')


In [12]:

# with open(model_file, "rb") as f:
#     base_me = pickle.load(f)
    
base_expressions = compile_expressions(base_me)
base_expressions

{(None, 0): (<function _lambdifygenerated(mu)>,
  <function _lambdifygenerated(mu)>),
 (6545, 348): <function _lambdifygenerated(mu)>,
 (6492, 348): <function _lambdifygenerated(mu)>,
 (5231, 348): <function _lambdifygenerated(mu)>,
 (5839, 348): <function _lambdifygenerated(mu)>,
 (5398, 348): <function _lambdifygenerated(mu)>,
 (5445, 348): <function _lambdifygenerated(mu)>,
 (6495, 348): <function _lambdifygenerated(mu)>,
 (6500, 348): <function _lambdifygenerated(mu)>,
 (6545, 349): <function _lambdifygenerated(mu)>,
 (6492, 349): <function _lambdifygenerated(mu)>,
 (5231, 349): <function _lambdifygenerated(mu)>,
 (5839, 349): <function _lambdifygenerated(mu)>,
 (5398, 349): <function _lambdifygenerated(mu)>,
 (5445, 349): <function _lambdifygenerated(mu)>,
 (6495, 349): <function _lambdifygenerated(mu)>,
 (6500, 349): <function _lambdifygenerated(mu)>,
 (6545, 350): <function _lambdifygenerated(mu)>,
 (6492, 350): <function _lambdifygenerated(mu)>,
 (5231, 350): <function _lambdif

In [13]:
uptake_df = pd.read_csv('fluxes/imod_constrained_uptakes.csv',index_col='nitrogen_source')
uptake_df

,growth_rate,glucose_exchange,glucose_uptake,n_source_exchange,n_source_uptake,n_source_uptake_lb,N_uptake,C_uptake_total
nitrogen_source,,,,,,,,
NH4Cl,0.7415,EX_glc__D_e,9.346562,EX_nh4_e,8.000000,8.000000,8.000000,32.544834
Adenine,0.5530,EX_glc__D_e,7.782946,EX_ade_e,1.186681,1.600000,5.933405,24.130288
L-Cys,0.1240,EX_glc__D_e,5.354672,EX_cys__L_e,1.344798,8.000000,1.344798,5.415969
Cytosine,0.2485,EX_glc__D_e,6.738419,EX_csn_e,0.870683,2.666667,2.612050,13.231134
Cytidine,0.4870,EX_glc__D_e,10.000000,EX_cytd_e,1.732899,2.666667,5.198696,25.944337
Putrescine,0.4135,EX_glc__D_e,9.860923,EX_ptrc_e,2.207418,4.000000,4.414835,18.038417
L-Pro,0.1350,EX_glc__D_e,5.919111,EX_pro__L_e,1.460313,8.000000,1.460313,5.905605
Adenosine,0.6260,EX_glc__D_e,7.033991,EX_adn_e,1.356117,1.600000,6.780583,27.482723
L-Asn,0.4185,EX_glc__D_e,9.087707,EX_asn__L_e,2.256225,4.000000,4.512449,18.402628


In [14]:
from cobrame.solve.algorithms import binary_search

# fix each condition to the uptakes from the min-carbon solution,
# then let mu float: no sector and no growth-rate constraints
max_growth = {}
uptake_flux_dict = {}
uptake_translation_flux_dict = {}
tol = 1e-6

for nitrogen_source, row in uptake_df.iterrows():

    print("\n", nitrogen_source)

    me = pickle.loads(pickle.dumps(base_me))

    for ex in nitrogen_exchange_map.values():
        if ex in me.reactions:
            me.reactions.get_by_id(ex).lower_bound = 0

    # lb = ub = -uptake pins the substrate to exactly that rate.
    # if binary_search raises "0 needs to be feasible", set the
    # upper bound to 0 instead to make it an upper limit.
    for rid, uptake in [
        ("EX_glc__D_e", row["glucose_uptake"]),
        (row["n_source_exchange"], row["n_source_uptake"]),
    ]:
        rxn = me.reactions.get_by_id(rid)
        # band of +/- tol AROUND -uptake. the signs matter: writing
        # -(uptake-tol) as the lower bound puts it ABOVE the upper
        # bound, giving an empty box and an LP infeasible at every mu
        rxn.lower_bound = -uptake
        rxn.upper_bound = 0

    binary_search(
        me,
        min_mu=0.1,
        max_mu=0.9,
        mu_accuracy=1e-2,
        solver="soplex",
        compiled_expressions=base_expressions,
        verbose=True
    )

    max_growth[nitrogen_source] = me.solution.f

    # binary_search leaves me.solution at the last feasible mu
    uptake_flux_dict[nitrogen_source] = (
        me.solution.x_dict.copy()
    )

    uptake_translation_flux_dict[nitrogen_source] = (
        me.get_translation_flux().copy()
    )

    print("mu_max =", me.solution.f, "| mu_obs =", row["growth_rate"])

max_growth_df = pd.DataFrame({
    "mu_obs": uptake_df["growth_rate"],
    "mu_max_at_fixed_uptake": pd.Series(max_growth),
})

uptake_fluxes = pd.DataFrame(
    uptake_flux_dict
)

uptake_translation_fluxes = pd.DataFrame(
    uptake_translation_flux_dict
)

max_growth_df.to_csv('fluxes/uptake_constrained_growth.csv')
uptake_fluxes.to_csv('fluxes/uptake_constrained_fluxes.csv')
uptake_translation_fluxes.to_csv('fluxes/uptake_constrained_translation_fluxes.csv')

max_growth_df



 NH4Cl
mu   	status
0.100	+
0.900	-
0.500	+
0.700	+
0.800	-
0.750	+
0.775	+
0.788	+
0.794	-
0.788	+
completed in 280.5 seconds and 10 iterations
mu_max = 0.7875000000000001 | mu_obs = 0.7414999999999999

 Adenine
mu   	status
0.100	+
0.900	-
0.500	+
0.700	+
0.800	-
0.750	-
0.725	-
0.712	-
0.706	+
0.706	+
completed in 1560.5 seconds and 10 iterations
mu_max = 0.7062499999999999 | mu_obs = 0.5529999999999999

 L-Cys
mu   	status
0.100	+
0.900	-
0.500	-
0.300	-
0.200	+
0.250	+
0.275	+
0.287	+
0.294	+
0.294	+
completed in 324.1 seconds and 10 iterations
mu_max = 0.29374999999999996 | mu_obs = 0.124

 Cytosine
mu   	status
0.100	+
0.900	-
0.500	-
0.300	+
0.400	+
0.450	+
0.475	-
0.463	-
0.456	-
0.450	+
completed in 2367.4 seconds and 10 iterations
mu_max = 0.45 | mu_obs = 0.2485

 Cytidine
mu   	status
0.100	+
0.900	-
0.500	+
0.700	-
0.600	+
0.650	+
0.675	-
0.662	-
0.656	+
0.656	+
completed in 747.3 seconds and 10 iterations
mu_max = 0.6562499999999999 | mu_obs = 0.48700000000000004

 Putre

,mu_obs,mu_max_at_fixed_uptake
nitrogen_source,,
NH4Cl,0.7415,0.78750
Adenine,0.5530,0.70625
L-Cys,0.1240,0.29375
Cytosine,0.2485,0.45000
Cytidine,0.4870,0.65625
Putrescine,0.4135,0.60625
L-Pro,0.1350,0.31250
Adenosine,0.6260,0.75625
L-Asn,0.4185,0.61875
